In [1]:
import cv2
import numpy as np
import torch
from tensorflow.keras.models import load_model

def detect_faces(image, face_model, face_labels):
    """
    Detect faces in the image using the face detection model.
    """
    face_cascade = cv2.CascadeClassifier(cv2.data.haarcascades + 'haarcascade_frontalface_default.xml')
    gray_image = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
    faces = face_cascade.detectMultiScale(gray_image, scaleFactor=1.3, minNeighbors=5, minSize=(30, 30))
    face_detections = []

    for (x, y, w, h) in faces:
        face_roi = image[y:y+h, x:x+w]
        face_resized = cv2.resize(face_roi, (100, 100))
        face_resized = cv2.cvtColor(face_resized, cv2.COLOR_BGR2RGB)
        face_input = np.expand_dims(face_resized, axis=0) / 255.0

        # Predict face class
        predictions = face_model.predict(face_input)
        class_index = np.argmax(predictions)
        label = list(face_labels.keys())[class_index]
        confidence = predictions[0][class_index]

        # Append bounding box and label
        face_detections.append(((x, y, w, h), label, confidence))

    return face_detections


def detect_weapons1(image, yolo_model):
    """
    Detect weapons in the image using the YOLO model.
    """
    results = yolo_model(image)
    weapon_detections = []

    for result in results.xyxy[0]:  # Each result contains [x1, y1, x2, y2, confidence, class]
        x1, y1, x2, y2, conf, cls = result.tolist()
        weapon_detections.append(((int(x1), int(y1), int(x2), int(y2)), conf))

    return weapon_detections

def detect_weapons(image, yolo_model):
    """
    Detect weapons in the image using the YOLO model.
    """
    results = yolo_model(image)  # Run inference on the image
    weapon_detections = []

    # Access detected boxes
    for box in results[0].boxes:  # Iterate through boxes in the first result (single image)
        # Extract bounding box coordinates, confidence score, and class
        x1, y1, x2, y2 = box.xyxy[0].cpu().numpy()  # Bounding box coordinates
        conf = box.conf[0].cpu().numpy()  # Confidence score
        cls = int(box.cls[0].cpu().numpy())  # Class ID

        # Add detection to list (optional: filter by class ID for weapons if needed)
        weapon_detections.append(((int(x1), int(y1), int(x2), int(y2)), conf))

    return weapon_detections

def integrate_detections(image, face_model, face_labels, yolo_model):
    """
    Integrate face detection and YOLO-based weapon detection into a single pipeline.
    """
    # Detect faces
    face_detections = detect_faces(image, face_model, face_labels)

    # Detect weapons
    weapon_detections = detect_weapons(image, yolo_model)

    # Draw face detections
    for (bbox, label, confidence) in face_detections:
        x, y, w, h = bbox
        cv2.rectangle(image, (x, y), (x+w, y+h), (0, 255, 0), 2)  # Green box
        cv2.putText(image, f"{label} ({confidence:.2f})", (x, y-10), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 255, 0), 2)

    # Draw weapon detections
    for (bbox, confidence) in weapon_detections:
        x1, y1, x2, y2 = bbox
        cv2.rectangle(image, (x1, y1), (x2, y2), (0, 0, 255), 2)  # Red box
        cv2.putText(image, f"Weapon ({confidence:.2f})", (x1, y1-10), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 0, 255), 2)

    return image


# Example usage

from ultralytics import YOLO
# Load the face detection model
face_model_path = "face_recognition.h5"
face_model = load_model(face_model_path)
# Load face class labels
face_labels = np.load('class_indices.npy', allow_pickle=True).item()

    # Load the YOLO model
    #yolo_model = torch.hub.load('ultralytics/yolov5', 'custom', path='best.pt', force_reload=True)
yolo_model = YOLO(r"C:\Users\hemas\Desktop\major proj\weapon_detection\weapon_detection\best.pt")
#yolo_model = YOLO("best.pt")
    # Input image
input_image_path = r"C:\Users\hemas\Desktop\bashi_images\two.jpg"
image = cv2.imread(input_image_path)

    # Perform detection
output_image = integrate_detections(image, face_model, face_labels, yolo_model)

    # Save and display output
output_image_path = "output_image_2_bashi.jpg"
cv2.imwrite(output_image_path, output_image)
print(f"Output saved at {output_image_path}")

1/1 ━━━━━━━━━━━━━━━━━━━━ 4s 4s/step

0: 640x480 1 eto, 345.0ms
Speed: 18.0ms preprocess, 345.0ms inference, 14.0ms postprocess per image at shape (1, 3, 640, 480)
Output saved at output_image_2_bashi.jpg


In [4]:
input_image_path = r"C:\Users\hemas\Desktop\bashi_images\two.jpg"
image = cv2.imread(input_image_path)

    # Perform detection
output_image = integrate_detections(image, face_model, face_labels, yolo_model)

    # Save and display output
output_image_path = "output_image_with_weapon_bashi.jpg"
cv2.imwrite(output_image_path, output_image)
print(f"Output saved at {output_image_path}")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 60ms/step

0: 640x480 1 eto, 153.0ms
Speed: 3.0ms preprocess, 153.0ms inference, 1.0ms postprocess per image at shape (1, 3, 640, 480)
Output saved at output_image_with_weapon_bashi.jpg


In [14]:
input_image_path = r"C:\Users\hemas\Desktop\major proj\weapon_detection\weapon_detection\img2.jpg"
image = cv2.imread(input_image_path)

    # Perform detection
output_image = integrate_detections(image, face_model, face_labels, yolo_model)

    # Save and display output
output_image_path = "1.jpg"
cv2.imwrite(output_image_path, output_image)
print(f"Output saved at {output_image_path}")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 67ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 58ms/step

0: 640x288 (no detections), 136.9ms
Speed: 3.0ms preprocess, 136.9ms inference, 0.0ms postprocess per image at shape (1, 3, 640, 288)
Output saved at 1.jpg


In [5]:
input_image_path = r"C:\Users\hemas\Desktop\hema_images\two.jpg"
image = cv2.imread(input_image_path)

    # Perform detection
output_image = integrate_detections(image, face_model, face_labels, yolo_model)

    # Save and display output
output_image_path = "output_image_with_weapon_hema.jpg"
cv2.imwrite(output_image_path, output_image)
print(f"Output saved at {output_image_path}")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 55ms/step

0: 640x480 (no detections), 135.0ms
Speed: 3.0ms preprocess, 135.0ms inference, 0.0ms postprocess per image at shape (1, 3, 640, 480)
Output saved at output_image_with_weapon_hema.jpg


In [13]:
input_image_path = r"C:\Users\hemas\Desktop\siva_images\one.jpg"
image = cv2.imread(input_image_path)

    # Perform detection
output_image = integrate_detections(image, face_model, face_labels, yolo_model)

    # Save and display output
output_image_path = "output_image_with_weapon_siva.jpg"
cv2.imwrite(output_image_path, output_image)
print(f"Output saved at {output_image_path}")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 65ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 57ms/step

0: 640x480 (no detections), 145.6ms
Speed: 3.0ms preprocess, 145.6ms inference, 0.0ms postprocess per image at shape (1, 3, 640, 480)
Output saved at output_image_with_weapon_siva.jpg


In [7]:
input_image_path = r"C:\Users\hemas\Desktop\thanuja_images\two.jpg"
image = cv2.imread(input_image_path)

    # Perform detection
output_image = integrate_detections(image, face_model, face_labels, yolo_model)

    # Save and display output
output_image_path = "output_image_with_weapon_thanuja.jpg"
cv2.imwrite(output_image_path, output_image)
print(f"Output saved at {output_image_path}")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 55ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 61ms/step

0: 640x480 (no detections), 134.0ms
Speed: 3.0ms preprocess, 134.0ms inference, 1.0ms postprocess per image at shape (1, 3, 640, 480)
Output saved at output_image_with_weapon_thanuja.jpg


In [6]:
input_image_path = r"C:\Users\hemas\Desktop\D7-DVD\weapon_detection\single_with_weapon.jpg"
image = cv2.imread(input_image_path)

    # Perform detection
output_image = integrate_detections(image, face_model, face_labels, yolo_model)

    # Save and display output
output_image_path = "output_image_bashi.jpg"
cv2.imwrite(output_image_path, output_image)
print(f"Output saved at {output_image_path}")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 121ms/step

0: 640x288 (no detections), 241.0ms
Speed: 4.0ms preprocess, 241.0ms inference, 1.0ms postprocess per image at shape (1, 3, 640, 288)
Output saved at output_image_bashi.jpg


In [1]:
import cv2
import mediapipe as mp

# Initialize MediaPipe Pose model
mp_pose = mp.solutions.pose
pose = mp_pose.Pose(min_detection_confidence=0.5, min_tracking_confidence=0.5)
def integrate_detections(image, face_model, face_labels, yolo_model):
    """
    Integrates face detection, YOLO-based weapon detection, and pose estimation into one pipeline.
    """
    # Detect faces and weapons using your existing methods
    # Assuming 'integrate_detections' includes face detection and weapon detection (from your provided code)
    image_with_faces_and_weapons = integrate_detections(image, face_model, face_labels, yolo_model)
    
    # Perform pose estimation on the image
    image_with_pose, pose_landmarks = detect_pose(image_with_faces_and_weapons)
    
    if pose_landmarks:
        # Draw pose landmarks on the image
        for landmark in pose_landmarks.landmark:
            # Access the x, y, z values
            x, y, z = landmark.x, landmark.y, landmark.z
            # Convert normalized coordinates to image dimensions
            x = int(x * image.shape[1])
            y = int(y * image.shape[0])
            # Draw the keypoints on the image (blue dots for pose landmarks)
            cv2.circle(image_with_pose, (x, y), 5, (255, 0, 0), -1)
    
    return image_with_pose

def detect_pose(image):
    """
    Detects pose keypoints (body landmarks) in the input image using MediaPipe Pose.
    """
    # Convert the image to RGB (MediaPipe uses RGB format)
    image_rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    
    # Perform pose estimation
    results = pose.process(image_rgb)
    
    # Check if landmarks are detected
    if results.pose_landmarks:
        # Loop through the detected landmarks and annotate them
        for landmark in results.pose_landmarks.landmark:
            # Get coordinates of the landmarks
            h, w, _ = image.shape
            x, y = int(landmark.x * w), int(landmark.y * h)
            # Draw the landmarks on the image (you can adjust the radius and color)
            cv2.circle(image, (x, y), 5, (0, 255, 0), -1)
        
        # Return the image with keypoints drawn
        return image, results.pose_landmarks
    else:
        # Return original image if no pose landmarks were detected
        return image, None


def integrate_pose_estimation(image, face_model, face_labels, yolo_model):
    """
    Integrate face detection, YOLO-based weapon detection, and pose estimation into a single pipeline.
    """
    # Detect faces and weapons using your existing methods
    image_with_faces_and_weapons = integrate_detections(image, face_model, face_labels, yolo_model)

    # Perform pose estimation on the image
    image_with_pose, pose_landmarks = detect_pose(image_with_faces_and_weapons)

    # Draw pose landmarks on the image
    if pose_landmarks:
        for landmark in pose_landmarks.landmark:
            x, y = int(landmark.x * image.shape[1]), int(landmark.y * image.shape[0])
            cv2.circle(image_with_pose, (x, y), 5, (255, 0, 0), -1)  # Blue color for pose landmarks

    return image_with_pose


In [7]:
# Example usage
image = cv2.imread(r"C:\Users\hemas\Desktop\major proj\weapon_detection\weapon_detection\img2.jpg")

# Integrate face detection, weapon detection, and pose estimation
output_image = integrate_pose_estimation(image, face_model, face_labels, yolo_model)

# Save or display the output image
cv2.imshow('Output Image1', output_image)
cv2.imwrite('output_image_with_pose1.jpg', output_image)
cv2.waitKey(0)
cv2.destroyAllWindows()


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 118ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 117ms/step

0: 640x288 (no detections), 159.0ms
Speed: 5.0ms preprocess, 159.0ms inference, 1.0ms postprocess per image at shape (1, 3, 640, 288)


In [ ]:
a

In [10]:
import cv2
import numpy as np
import torch
import mediapipe as mp
from tensorflow.keras.models import load_model
from scipy.spatial.distance import euclidean

# MediaPipe Hand Detection Setup
mp_hands = mp.solutions.hands
hands = mp_hands.Hands(min_detection_confidence=0.7, min_tracking_confidence=0.7)

def detect_hands(image):
    """
    Detect hands and return the coordinates of the left and right wrists.
    """
    # Convert the image to RGB (MediaPipe works with RGB)
    rgb_image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    results = hands.process(rgb_image)

    wrist_positions = []
    
    if results.multi_hand_landmarks:
        for hand_landmarks in results.multi_hand_landmarks:
            # Get the left wrist (landmark 9) and right wrist (landmark 12)
            left_wrist = hand_landmarks.landmark[mp_hands.HandLandmark.WRIST]
            wrist_positions.append((left_wrist.x, left_wrist.y))
            
    return wrist_positions

def detect_faces(image, face_model, face_labels):
    """
    Detect faces in the image using the face detection model.
    """
    face_cascade = cv2.CascadeClassifier(cv2.data.haarcascades + 'haarcascade_frontalface_default.xml')
    gray_image = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
    faces = face_cascade.detectMultiScale(gray_image, scaleFactor=1.3, minNeighbors=5, minSize=(30, 30))
    face_detections = []

    for (x, y, w, h) in faces:
        face_roi = image[y:y+h, x:x+w]
        face_resized = cv2.resize(face_roi, (100, 100))
        face_resized = cv2.cvtColor(face_resized, cv2.COLOR_BGR2RGB)
        face_input = np.expand_dims(face_resized, axis=0) / 255.0

        # Predict face class
        predictions = face_model.predict(face_input)
        class_index = np.argmax(predictions)
        label = list(face_labels.keys())[class_index]
        confidence = predictions[0][class_index]

        # Append bounding box and label
        face_detections.append(((x, y, w, h), label, confidence))

    return face_detections


def detect_weapons(image, yolo_model):
    """
    Detect weapons in the image using the YOLO model.
    """
    results = yolo_model(image)  # Run inference on the image
    weapon_detections = []

    # Access detected boxes
    for box in results[0].boxes:  # Iterate through boxes in the first result (single image)
        # Extract bounding box coordinates, confidence score, and class
        x1, y1, x2, y2 = box.xyxy[0].cpu().numpy()  # Bounding box coordinates
        conf = box.conf[0].cpu().numpy()  # Confidence score
        cls = int(box.cls[0].cpu().numpy())  # Class ID

        # Add detection to list (optional: filter by class ID for weapons if needed)
        weapon_detections.append(((int(x1), int(y1), int(x2), int(y2)), conf))

    return weapon_detections

def integrate_detections(image, face_model, face_labels, yolo_model):
    """
    Integrate face detection, hand detection, and YOLO-based weapon detection into a single pipeline.
    """
    # Detect faces
    face_detections = detect_faces(image, face_model, face_labels)

    # Detect weapons
    weapon_detections = detect_weapons(image, yolo_model)

    # Detect hands
    wrist_positions = detect_hands(image)

    # Calculate distances between hands and weapons
    for wrist in wrist_positions:
        wrist_x, wrist_y = wrist

        for (bbox, conf) in weapon_detections:
            x1, y1, x2, y2 = bbox
            weapon_center_x = (x1 + x2) / 2
            weapon_center_y = (y1 + y2) / 2

            # Calculate Euclidean distance between the wrist and the weapon center
            distance = euclidean((wrist_x, wrist_y), (weapon_center_x, weapon_center_y))

            # Draw distance info on the image
            cv2.putText(image, f"Distance: {distance:.2f}", (int(wrist_x * image.shape[1]), int(wrist_y * image.shape[0])),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.8, (255, 255, 0), 2)

    # Draw face detections
    for (bbox, label, confidence) in face_detections:
        x, y, w, h = bbox
        cv2.rectangle(image, (x, y), (x+w, y+h), (0, 255, 0), 2)  # Green box
        cv2.putText(image, f"{label} ({confidence:.2f})", (x, y-10), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 255, 0), 2)

    # Draw weapon detections
    for (bbox, confidence) in weapon_detections:
        x1, y1, x2, y2 = bbox
        cv2.rectangle(image, (x1, y1), (x2, y2), (0, 0, 255), 2)  # Red box
        cv2.putText(image, f"Weapon ({confidence:.2f})", (x1, y1-10), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 0, 255), 2)

    return image


# Example usage

from ultralytics import YOLO
# Load the face detection model
face_model_path = "facerecognition_finalmodel.h5"
face_model = load_model(face_model_path)
# Load face class labels
face_labels = np.load('class_indices.npy', allow_pickle=True).item()

# Load the YOLO model
yolo_model = YOLO(r"C:\Users\hemas\Desktop\major proj\weapon_detection\weapon_detection\best.pt")

# Input image
input_image_path = r"C:\Users\hemas\Desktop\D7-DVD\weapon_detection\siva_images\one.jpg"
image = cv2.imread(input_image_path)

# Perform detection
output_image = integrate_detections(image, face_model, face_labels, yolo_model)

# Save and display output
output_image_path = "output_image5.jpg"
cv2.imwrite(output_image_path, output_image)
print(f"Output saved at {output_image_path}")


1/1 ━━━━━━━━━━━━━━━━━━━━ 3s 3s/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 102ms/step

0: 640x480 (no detections), 226.0ms
Speed: 7.0ms preprocess, 226.0ms inference, 1.0ms postprocess per image at shape (1, 3, 640, 480)
Output saved at output_image5.jpg


In [4]:
import cv2
import numpy as np
import torch
from tensorflow.keras.models import load_model
from ultralytics import YOLO

def detect_faces(image, face_model, face_labels):
    """
    Detect faces in the image using the face detection model.
    """
    face_cascade = cv2.CascadeClassifier(cv2.data.haarcascades + 'haarcascade_frontalface_default.xml')
    gray_image = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
    faces = face_cascade.detectMultiScale(gray_image, scaleFactor=1.3, minNeighbors=5, minSize=(30, 30))
    face_detections = []

    for (x, y, w, h) in faces:
        face_roi = image[y:y+h, x:x+w]
        face_resized = cv2.resize(face_roi, (100, 100))
        face_resized = cv2.cvtColor(face_resized, cv2.COLOR_BGR2RGB)
        face_input = np.expand_dims(face_resized, axis=0) / 255.0

        # Predict face class
        predictions = face_model.predict(face_input)
        class_index = np.argmax(predictions)
        label = list(face_labels.keys())[class_index]
        confidence = predictions[0][class_index]

        # Append bounding box, label, and confidence
        face_detections.append(((x, y, w, h), label, confidence))

    return face_detections

def detect_weapons(image, yolo_model):
    """
    Detect weapons in the image using the YOLO model.
    """
    results = yolo_model(image)  # Run inference on the image
    weapon_detections = []

    # Access detected boxes
    for box in results[0].boxes:  # Iterate through boxes in the first result (single image)
        x1, y1, x2, y2 = box.xyxy[0].cpu().numpy()  # Bounding box coordinates
        conf = box.conf[0].cpu().numpy()  # Confidence score
        cls = int(box.cls[0].cpu().numpy())  # Class ID

        # Add detection to list (optional: filter by class ID for weapons if needed)
        weapon_detections.append(((int(x1), int(y1), int(x2), int(y2)), conf))

    return weapon_detections

def integrate_detections(image, face_model, face_labels, yolo_model):
    """
    Integrate face detection and YOLO-based weapon detection into a single pipeline.
    """
    # Detect faces
    face_detections = detect_faces(image, face_model, face_labels)

    # Detect weapons
    weapon_detections = detect_weapons(image, yolo_model)

    # Draw face detections with name
    for (bbox, label, confidence) in face_detections:
        x, y, w, h = bbox
        cv2.rectangle(image, (x, y), (x+w, y+h), (0, 255, 0), 2)  # Green box
        cv2.putText(image, f"{label} ({confidence:.2f})", (x, y-10), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 255, 0), 2)

    # Draw weapon detections
    for (bbox, confidence) in weapon_detections:
        x1, y1, x2, y2 = bbox
        cv2.rectangle(image, (x1, y1), (x2, y2), (0, 0, 255), 2)  # Red box
        cv2.putText(image, f"Weapon ({confidence:.2f})", (x1, y1-10), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 0, 255), 2)

    return image

# Example usage
# Load the face detection model
face_model_path = "facerecognition_finalmodel.h5"
face_model = load_model(face_model_path)
# Load face class labels
face_labels = np.load('class_indices.npy', allow_pickle=True).item()

# Load the YOLO model for weapon detection
yolo_model = YOLO(r"D:\wepon_detection\runs\detect\train12\weights\best.pt")

# Input image
input_image_path = "single_with_weapon.jpeg"
image = cv2.imread(input_image_path)

# Perform detection
output_image = integrate_detections(image, face_model, face_labels, yolo_model)

# Save and display output
output_image_path = "output_image.jpg"
cv2.imwrite(output_image_path, output_image)
print(f"Output saved at {output_image_path}")


1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 976ms/step

0: 640x288 1 weapon, 83.2ms
Speed: 10.3ms preprocess, 83.2ms inference, 0.0ms postprocess per image at shape (1, 3, 640, 288)
Output saved at output_image.jpg


In [6]:
import cv2
import numpy as np
import torch
import mediapipe as mp
from tensorflow.keras.models import load_model
from scipy.spatial.distance import euclidean

# MediaPipe Hand Detection Setup
mp_hands = mp.solutions.hands
hands = mp_hands.Hands(min_detection_confidence=0.7, min_tracking_confidence=0.7)

def detect_hands(image):
    """
    Detect hands and return the coordinates of the left and right wrists.
    """
    # Convert the image to RGB (MediaPipe works with RGB)
    rgb_image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    results = hands.process(rgb_image)

    wrist_positions = []
    
    if results.multi_hand_landmarks:
        for hand_landmarks in results.multi_hand_landmarks:
            # Get the left wrist (landmark 9) and right wrist (landmark 12)
            left_wrist = hand_landmarks.landmark[mp_hands.HandLandmark.WRIST]
            wrist_positions.append((left_wrist.x, left_wrist.y))
            
    return wrist_positions

def detect_faces(image, face_model, face_labels):
    """
    Detect faces in the image using the face detection model.
    """
    face_cascade = cv2.CascadeClassifier(cv2.data.haarcascades + 'haarcascade_frontalface_default.xml')
    gray_image = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
    faces = face_cascade.detectMultiScale(gray_image, scaleFactor=1.3, minNeighbors=5, minSize=(30, 30))
    face_detections = []

    for (x, y, w, h) in faces:
        face_roi = image[y:y+h, x:x+w]
        face_resized = cv2.resize(face_roi, (100, 100))
        face_resized = cv2.cvtColor(face_resized, cv2.COLOR_BGR2RGB)
        face_input = np.expand_dims(face_resized, axis=0) / 255.0

        # Predict face class
        predictions = face_model.predict(face_input)
        class_index = np.argmax(predictions)
        label = list(face_labels.keys())[class_index]
        confidence = predictions[0][class_index]

        # Append bounding box and label
        face_detections.append(((x, y, w, h), label, confidence))

    return face_detections


def detect_weapons(image, yolo_model):
    """
    Detect weapons in the image using the YOLO model.
    """
    results = yolo_model(image)  # Run inference on the image
    weapon_detections = []

    # Access detected boxes
    for box in results[0].boxes:  # Iterate through boxes in the first result (single image)
        # Extract bounding box coordinates, confidence score, and class
        x1, y1, x2, y2 = box.xyxy[0].cpu().numpy()  # Bounding box coordinates
        conf = box.conf[0].cpu().numpy()  # Confidence score
        cls = int(box.cls[0].cpu().numpy())  # Class ID

        # Add detection to list (optional: filter by class ID for weapons if needed)
        weapon_detections.append(((int(x1), int(y1), int(x2), int(y2)), conf))

    return weapon_detections

def integrate_detections(image, face_model, face_labels, yolo_model):
    """
    Integrate face detection, hand detection, and YOLO-based weapon detection into a single pipeline.
    """
    # Detect faces
    face_detections = detect_faces(image, face_model, face_labels)

    # Detect weapons
    weapon_detections = detect_weapons(image, yolo_model)

    # Detect hands
    wrist_positions = detect_hands(image)

    # Process each wrist and weapon pair
    for wrist in wrist_positions:
        wrist_x, wrist_y = wrist

        for (bbox, conf) in weapon_detections:
            x1, y1, x2, y2 = bbox
            weapon_center_x = (x1 + x2) / 2
            weapon_center_y = (y1 + y2) / 2

            # Calculate Euclidean distance between the wrist and the weapon center
            distance = euclidean((wrist_x, wrist_y), (weapon_center_x, weapon_center_y))

            # Draw distance info on the image
            cv2.putText(image, f"Distance: {distance:.2f}", 
                        (int(wrist_x * image.shape[1]), int(wrist_y * image.shape[0])),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.8, (255, 255, 0), 2)

    # Draw face detections with name and distance
    for (bbox, label, confidence) in face_detections:
        x, y, w, h = bbox
        cv2.rectangle(image, (x, y), (x+w, y+h), (0, 255, 0), 2)  # Green box
        cv2.putText(image, f"{label} ({confidence:.2f})", 
                    (x, y-10), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 255, 0), 2)

        # Optionally, print name and distance near face
        for wrist in wrist_positions:
            wrist_x, wrist_y = wrist
            face_center_x = x + w / 2
            face_center_y = y + h / 2
            # Calculate distance between wrist and face center
            distance = euclidean((wrist_x, wrist_y), (face_center_x, face_center_y))
            cv2.putText(image, f"Distance to {label}: {distance:.2f}", 
                        (x, y-30), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 255, 255), 2)

    # Draw weapon detections
    for (bbox, confidence) in weapon_detections:
        x1, y1, x2, y2 = bbox
        cv2.rectangle(image, (x1, y1), (x2, y2), (0, 0, 255), 2)  # Red box
        cv2.putText(image, f"Weapon ({confidence:.2f})", 
                    (x1, y1-10), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 0, 255), 2)

    return image


# Example usage

from ultralytics import YOLO
# Load the face detection model
face_model_path = "facerecognition_finalmodel.h5"
face_model = load_model(face_model_path)
# Load face class labels
face_labels = np.load('class_indices.npy', allow_pickle=True).item()

# Load the YOLO model
yolo_model = YOLO(r"D:\wepon_detection\runs\detect\train12\weights\best.pt")

# Input image
input_image_path = "double_with_weapon.jpeg"
image = cv2.imread(input_image_path)

# Perform detection
output_image = integrate_detections(image, face_model, face_labels, yolo_model)

# Save and display output
output_image_path = "output_image.jpg"
cv2.imwrite(output_image_path, output_image)
print(f"Output saved at {output_image_path}")


1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 1s/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 35ms/step

0: 288x640 (no detections), 90.6ms
Speed: 2.5ms preprocess, 90.6ms inference, 0.0ms postprocess per image at shape (1, 3, 288, 640)
Output saved at output_image.jpg


In [9]:
import cv2
import numpy as np
import torch
import mediapipe as mp
from tensorflow.keras.models import load_model
from scipy.spatial.distance import euclidean

# MediaPipe Hand Detection Setup
mp_hands = mp.solutions.hands
hands = mp_hands.Hands(min_detection_confidence=0.7, min_tracking_confidence=0.7)

def detect_hands(image):
    """
    Detect hands and return the coordinates of the left and right wrists.
    """
    # Convert the image to RGB (MediaPipe works with RGB)
    rgb_image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    results = hands.process(rgb_image)

    wrist_positions = []
    
    if results.multi_hand_landmarks:
        for hand_landmarks in results.multi_hand_landmarks:
            # Get the wrist (landmark 0)
            wrist = hand_landmarks.landmark[mp_hands.HandLandmark.WRIST]
            wrist_positions.append((wrist.x, wrist.y))
            
    return wrist_positions

def detect_faces(image, face_model, face_labels):
    """
    Detect faces in the image using the face detection model.
    """
    face_cascade = cv2.CascadeClassifier(cv2.data.haarcascades + 'haarcascade_frontalface_default.xml')
    gray_image = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
    faces = face_cascade.detectMultiScale(gray_image, scaleFactor=1.3, minNeighbors=5, minSize=(30, 30))
    face_detections = []

    for (x, y, w, h) in faces:
        face_roi = image[y:y+h, x:x+w]
        face_resized = cv2.resize(face_roi, (100, 100))
        face_resized = cv2.cvtColor(face_resized, cv2.COLOR_BGR2RGB)
        face_input = np.expand_dims(face_resized, axis=0) / 255.0

        # Predict face class
        predictions = face_model.predict(face_input)
        class_index = np.argmax(predictions)
        label = list(face_labels.keys())[class_index]
        confidence = predictions[0][class_index]

        # Append bounding box and label
        face_detections.append(((x, y, w, h), label, confidence))

    return face_detections


def detect_weapons(image, yolo_model):
    """
    Detect weapons in the image using the YOLO model.
    """
    results = yolo_model(image)  # Run inference on the image
    weapon_detections = []

    # Access detected boxes
    for box in results[0].boxes:  # Iterate through boxes in the first result (single image)
        # Extract bounding box coordinates, confidence score, and class
        x1, y1, x2, y2 = box.xyxy[0].cpu().numpy()  # Bounding box coordinates
        conf = box.conf[0].cpu().numpy()  # Confidence score
        cls = int(box.cls[0].cpu().numpy())  # Class ID

        # Add detection to list (optional: filter by class ID for weapons if needed)
        weapon_detections.append(((int(x1), int(y1), int(x2), int(y2)), conf))

    return weapon_detections

def integrate_detections(image, face_model, face_labels, yolo_model):
    """
    Integrate face detection, hand detection, and YOLO-based weapon detection into a single pipeline.
    """
    # Detect faces (optional, as you are focused on hands and weapons)
    face_detections = detect_faces(image, face_model, face_labels)

    # Detect weapons
    weapon_detections = detect_weapons(image, yolo_model)

    # Detect hands
    wrist_positions = detect_hands(image)

    # Process each weapon detection
    for (bbox, conf) in weapon_detections:
        x1, y1, x2, y2 = bbox
        weapon_center_x = (x1 + x2) / 2
        weapon_center_y = (y1 + y2) / 2

        # Find the nearest hand to the weapon
        min_distance = float('inf')
        nearest_wrist = None

        for wrist in wrist_positions:
            wrist_x, wrist_y = wrist
            # Calculate Euclidean distance between wrist and weapon center
            distance = euclidean((wrist_x, wrist_y), (weapon_center_x, weapon_center_y))

            if distance < min_distance:
                min_distance = distance
                nearest_wrist = wrist

        if nearest_wrist:
            wrist_x, wrist_y = nearest_wrist
            # Draw the nearest wrist distance info on the image
            cv2.putText(image, f"Distance: {min_distance:.2f}", 
                        (int(wrist_x * image.shape[1]), int(wrist_y * image.shape[0])),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.8, (255, 255, 0), 2)

        # Draw weapon detection
        cv2.rectangle(image, (x1, y1), (x2, y2), (0, 0, 255), 2)  # Red box
        cv2.putText(image, f"Weapon ({conf:.2f})", (x1, y1-10), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 0, 255), 2)

    # Draw face detections (optional)
    for (bbox, label, confidence) in face_detections:
        x, y, w, h = bbox
        cv2.rectangle(image, (x, y), (x+w, y+h), (0, 255, 0), 2)  # Green box
        cv2.putText(image, f"{label} ({confidence:.2f})", (x, y-10), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 255, 0), 2)

    return image


# Example usage

from ultralytics import YOLO
# Load the face detection model
face_model_path = "facerecognition_finalmodel.h5"
face_model = load_model(face_model_path)
# Load face class labels
face_labels = np.load('class_indices.npy', allow_pickle=True).item()

# Load the YOLO model
yolo_model = YOLO(r"D:\wepon_detection\runs\detect\train12\weights\best.pt")

# Input image
input_image_path = "single_with_weapon.jpeg"
image = cv2.imread(input_image_path)

# Perform detection
output_image = integrate_detections(image, face_model, face_labels, yolo_model)

# Save and display output
output_image_path = "output_image.jpg"
cv2.imwrite(output_image_path, output_image)
print(f"Output saved at {output_image_path}")


1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 927ms/step

0: 640x288 1 weapon, 80.0ms
Speed: 1.8ms preprocess, 80.0ms inference, 1.0ms postprocess per image at shape (1, 3, 640, 288)
Output saved at output_image.jpg


In [10]:
# Input image
input_image_path = "img3.jpeg"
image = cv2.imread(input_image_path)

# Perform detection
output_image = integrate_detections(image, face_model, face_labels, yolo_model)

# Save and display output
output_image_path = "output_image3.jpg"
cv2.imwrite(output_image_path, output_image)
print(f"Output saved at {output_image_path}")


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 40ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 34ms/step

0: 640x288 (no detections), 91.7ms
Speed: 2.0ms preprocess, 91.7ms inference, 1.0ms postprocess per image at shape (1, 3, 640, 288)
Output saved at output_image3.jpg


In [11]:
# Input image
input_image_path = "img4.jpeg"
image = cv2.imread(input_image_path)

# Perform detection
output_image = integrate_detections(image, face_model, face_labels, yolo_model)

# Save and display output
output_image_path = "output_image4.jpg"
cv2.imwrite(output_image_path, output_image)
print(f"Output saved at {output_image_path}")


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 37ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 35ms/step

0: 640x288 (no detections), 81.6ms
Speed: 3.0ms preprocess, 81.6ms inference, 0.0ms postprocess per image at shape (1, 3, 640, 288)
Output saved at output_image4.jpg


In [21]:
# Input image
input_image_path = "img2.jpg"
image = cv2.imread(input_image_path)

# Perform detection
output_image = integrate_detections(image, face_model, face_labels, yolo_model)

# Save and display output
output_image_path = "output_test.jpg"
cv2.imwrite(output_image_path, output_image)
print(f"Output saved at {output_image_path}")



0: 448x640 2 weapons, 119.6ms
Speed: 3.0ms preprocess, 119.6ms inference, 0.0ms postprocess per image at shape (1, 3, 448, 640)
Output saved at output_test.jpg


In [23]:
import cv2
import numpy as np
import torch
import mediapipe as mp
from tensorflow.keras.models import load_model
from scipy.spatial.distance import euclidean
from ultralytics import YOLO

# MediaPipe Setup
mp_hands = mp.solutions.hands
mp_pose = mp.solutions.pose
hands = mp_hands.Hands(min_detection_confidence=0.7, min_tracking_confidence=0.7)
pose = mp_pose.Pose(min_detection_confidence=0.7, min_tracking_confidence=0.7)

# Load the YOLO model for weapon detection
yolo_model = YOLO(r"D:\wepon_detection\runs\detect\train12\weights\best.pt")

# Load the face recognition model and labels
face_model_path = "facerecognition_finalmodel.h5"
face_model = load_model(face_model_path)
face_labels = np.load('class_indices.npy', allow_pickle=True).item()

def detect_hands(image):
    """
    Detect hands and return the wrist positions.
    """
    rgb_image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    results = hands.process(rgb_image)
    wrist_positions = []
    
    if results.multi_hand_landmarks:
        for hand_landmarks in results.multi_hand_landmarks:
            wrist = hand_landmarks.landmark[mp_hands.HandLandmark.WRIST]
            wrist_positions.append((wrist.x, wrist.y))
            
    return wrist_positions

def detect_pose(image):
    """
    Detect body pose keypoints and return pose landmarks.
    """
    rgb_image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    results = pose.process(rgb_image)
    pose_landmarks = []

    if results.pose_landmarks:
        for landmark in results.pose_landmarks.landmark:
            pose_landmarks.append((landmark.x, landmark.y))
    
    return pose_landmarks

def detect_faces(image):
    """
    Detect faces in the image using the face detection model.
    """
    face_cascade = cv2.CascadeClassifier(cv2.data.haarcascades + 'haarcascade_frontalface_default.xml')
    gray_image = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
    faces = face_cascade.detectMultiScale(gray_image, scaleFactor=1.3, minNeighbors=5, minSize=(30, 30))
    face_detections = []

    for (x, y, w, h) in faces:
        face_roi = image[y:y+h, x:x+w]
        face_resized = cv2.resize(face_roi, (100, 100))
        face_resized = cv2.cvtColor(face_resized, cv2.COLOR_BGR2RGB)
        face_input = np.expand_dims(face_resized, axis=0) / 255.0

        # Predict face class
        predictions = face_model.predict(face_input)
        class_index = np.argmax(predictions)
        label = list(face_labels.keys())[class_index]
        confidence = predictions[0][class_index]

        # Append detection with label and confidence
        face_detections.append(((x, y, w, h), label, confidence))

    return face_detections

def detect_weapons(image):
    """
    Detect weapons using YOLO model.
    """
    results = yolo_model(image)  # Run inference on the image
    weapon_detections = []

    for box in results[0].boxes:  # Iterate through boxes in the first result (single image)
        x1, y1, x2, y2 = box.xyxy[0].cpu().numpy()  # Bounding box coordinates
        conf = box.conf[0].cpu().numpy()  # Confidence score
        cls = int(box.cls[0].cpu().numpy())  # Class ID
        weapon_detections.append(((int(x1), int(y1), int(x2), int(y2)), conf))

    return weapon_detections

def integrate_detections(image):
    """
    Integrate face detection, hand detection, pose estimation, and weapon detection into one pipeline.
    """
    # Detect faces
    face_detections = detect_faces(image)

    # Detect weapons
    weapon_detections = detect_weapons(image)

    # Detect hands
    wrist_positions = detect_hands(image)

    # Detect pose (body keypoints)
    pose_landmarks = detect_pose(image)

    # Process each weapon detection
    for (bbox, conf) in weapon_detections:
        x1, y1, x2, y2 = bbox
        weapon_center_x = (x1 + x2) / 2
        weapon_center_y = (y1 + y2) / 2

        # Find the nearest wrist to the weapon
        min_distance = float('inf')
        nearest_wrist = None

        for wrist in wrist_positions:
            wrist_x, wrist_y = wrist
            distance = euclidean((wrist_x, wrist_y), (weapon_center_x, weapon_center_y))

            if distance < min_distance:
                min_distance = distance
                nearest_wrist = wrist

        if nearest_wrist:
            wrist_x, wrist_y = nearest_wrist
            cv2.putText(image, f"Distance: {min_distance:.2f}", 
                        (int(wrist_x * image.shape[1]), int(wrist_y * image.shape[0])),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.8, (255, 255, 0), 2)

        # Draw weapon detection
        cv2.rectangle(image, (x1, y1), (x2, y2), (0, 0, 255), 2)  # Red box
        cv2.putText(image, f"Weapon ({conf:.2f})", (x1, y1-10), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 0, 255), 2)

    # Draw face detections
    for (bbox, label, confidence) in face_detections:
        x, y, w, h = bbox
        cv2.rectangle(image, (x, y), (x+w, y+h), (0, 255, 0), 2)  # Green box
        cv2.putText(image, f"{label} ({confidence:.2f})", (x, y-10), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 255, 0), 2)

    # Optionally, draw pose landmarks
    for landmark in pose_landmarks:
        x, y = int(landmark[0] * image.shape[1]), int(landmark[1] * image.shape[0])
        cv2.circle(image, (x, y), 5, (0, 0, 255), -1)

    return image

# Example usage
input_image_path = "465679287-1617663771_jpeg_jpg.rf.81225b8bf367b2295c4adb571072a999.jpg"
image = cv2.imread(input_image_path)

output_image = integrate_detections(image)

# Save and display output
output_image_path = "output_image_with_pose.jpg"
cv2.imwrite(output_image_path, output_image)
print(f"Output saved at {output_image_path}")


1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 957ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 29ms/step

0: 640x640 1 weapon, 168.4ms
Speed: 1.6ms preprocess, 168.4ms inference, 15.6ms postprocess per image at shape (1, 3, 640, 640)
Output saved at output_image_with_pose.jpg


In [28]:
import cv2
import numpy as np
import mediapipe as mp
from tensorflow.keras.models import load_model
from scipy.spatial.distance import euclidean
from ultralytics import YOLO

# MediaPipe Setup
mp_pose = mp.solutions.pose
pose = mp_pose.Pose(min_detection_confidence=0.7, min_tracking_confidence=0.7)

# Load the YOLO model for weapon detection
yolo_model = YOLO(r"D:\wepon_detection\runs\detect\train12\weights\best.pt")

# Load the face recognition model and labels
face_model_path = "facerecognition_finalmodel.h5"
face_model = load_model(face_model_path)
face_labels = np.load('class_indices.npy', allow_pickle=True).item()

def detect_faces(image):
    """
    Detect faces and classify them using the face recognition model.
    """
    face_cascade = cv2.CascadeClassifier(cv2.data.haarcascades + 'haarcascade_frontalface_default.xml')
    gray_image = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
    faces = face_cascade.detectMultiScale(gray_image, scaleFactor=1.3, minNeighbors=5, minSize=(30, 30))
    face_detections = []

    for (x, y, w, h) in faces:
        face_roi = image[y:y+h, x:x+w]
        face_resized = cv2.resize(face_roi, (100, 100))
        face_resized = cv2.cvtColor(face_resized, cv2.COLOR_BGR2RGB)
        face_input = np.expand_dims(face_resized, axis=0) / 255.0

        # Predict face class
        predictions = face_model.predict(face_input)
        class_index = np.argmax(predictions)
        label = list(face_labels.keys())[class_index]
        confidence = predictions[0][class_index]

        # Append detection with label and confidence
        face_detections.append(((x, y, w, h), label, confidence))

    return face_detections

def detect_pose(image):
    """
    Detect body pose of a single person in the image.
    """
    rgb_image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    results = pose.process(rgb_image)

    if results.pose_landmarks:
        keypoints = [(lmk.x, lmk.y) for lmk in results.pose_landmarks.landmark]
        return [keypoints]
    return []

def detect_weapons(image):
    """
    Detect weapons in the image using YOLO.
    """
    results = yolo_model(image)
    weapon_detections = []

    for box in results[0].boxes:
        x1, y1, x2, y2 = box.xyxy[0].cpu().numpy()
        conf = box.conf[0].cpu().numpy()
        weapon_detections.append(((int(x1), int(y1), int(x2), int(y2)), conf))

    return weapon_detections

def assign_pose_to_faces(poses, face_detections, image_shape):
    """
    Assign the closest pose to each detected face based on proximity.
    """
    assigned_poses = []

    for face_bbox, label, _ in face_detections:
        x, y, w, h = face_bbox
        face_center = (x + w / 2, y + h / 2)
        min_distance = float('inf')
        assigned_pose = None

        for pose in poses:
            nose_x, nose_y = pose[mp_pose.PoseLandmark.NOSE.value]
            absolute_x = int(nose_x * image_shape[1])
            absolute_y = int(nose_y * image_shape[0])
            distance = euclidean(face_center, (absolute_x, absolute_y))

            if distance < min_distance:
                min_distance = distance
                assigned_pose = pose

        if assigned_pose:
            assigned_poses.append((label, assigned_pose))

    return assigned_poses

def determine_weapon_holder(assigned_poses, weapon_detections, image_shape):
    """
    Identify the person holding the weapon and which hand is holding it.
    """
    weapon_holders = []

    for label, pose in assigned_poses:
        right_wrist = pose[mp_pose.PoseLandmark.RIGHT_WRIST.value]
        left_wrist = pose[mp_pose.PoseLandmark.LEFT_WRIST.value]

        right_wrist_abs = (int(right_wrist[0] * image_shape[1]), int(right_wrist[1] * image_shape[0]))
        left_wrist_abs = (int(left_wrist[0] * image_shape[1]), int(left_wrist[1] * image_shape[0]))

        for weapon_bbox, _ in weapon_detections:
            x1, y1, x2, y2 = weapon_bbox
            weapon_center = ((x1 + x2) // 2, (y1 + y2) // 2)

            right_distance = euclidean(weapon_center, right_wrist_abs)
            left_distance = euclidean(weapon_center, left_wrist_abs)

            if right_distance < left_distance and right_distance < 50:  # Threshold distance
                weapon_holders.append((label, 'Right Hand'))
            elif left_distance < right_distance and left_distance < 50:  # Threshold distance
                weapon_holders.append((label, 'Left Hand'))

    return weapon_holders

def integrate_detections(image):
    """
    Integrate face detection, pose estimation, and weapon detection.
    """
    face_detections = detect_faces(image)
    poses = detect_pose(image)
    weapon_detections = detect_weapons(image)

    # Assign poses to faces
    assigned_poses = assign_pose_to_faces(poses, face_detections, image.shape)

    # Identify weapon holders
    weapon_holders = determine_weapon_holder(assigned_poses, weapon_detections, image.shape)
    print(weapon_holders)
    # Annotate the image
    for (bbox, label, _) in face_detections:
        x, y, w, h = bbox
        cv2.rectangle(image, (x, y), (x + w, y + h), (0, 255, 0), 2)
        cv2.putText(image, label, (x, y - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 255, 0), 2)

    for (bbox, _) in weapon_detections:
        x1, y1, x2, y2 = bbox
        cv2.rectangle(image, (x1, y1), (x2, y2), (0, 0, 255), 2)

    for label, hand in weapon_holders:
        cv2.putText(image, f"{label} ({hand})", (50, 50), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (255, 255, 0), 2)

    return image

# Example usage
input_image_path = "465679287-1617663771_jpeg_jpg.rf.81225b8bf367b2295c4adb571072a999.jpg"  # Replace with your image path
image = cv2.imread(input_image_path)
output_image = integrate_detections(image)
output_image_path = "output_image.jpg"
cv2.imwrite(output_image_path, output_image)


1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 927ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 47ms/step

0: 640x640 1 weapon, 152.7ms
Speed: 8.7ms preprocess, 152.7ms inference, 0.0ms postprocess per image at shape (1, 3, 640, 640)
[]


True

In [59]:
import cv2
import numpy as np
from mediapipe import solutions as mp_solutions
from scipy.spatial.distance import euclidean

mp_pose = mp_solutions.pose
mp_face_detection = mp_solutions.face_detection

def detect_faces(image):
    """Detect faces in the image using MediaPipe Face Detection."""
    with mp_face_detection.FaceDetection(model_selection=0, min_detection_confidence=0.5) as face_detection:
        results = face_detection.process(cv2.cvtColor(image, cv2.COLOR_BGR2RGB))
        faces = []
        if results.detections:
            for detection in results.detections:
                bbox = detection.location_data.relative_bounding_box
                x, y, w, h = (int(bbox.xmin * image.shape[1]),
                              int(bbox.ymin * image.shape[0]),
                              int(bbox.width * image.shape[1]),
                              int(bbox.height * image.shape[0]))
                faces.append(((x, y, w, h), f"Person_{len(faces) + 1}"))
        return faces

def detect_pose(image):
    """Detect pose landmarks in the image using MediaPipe Pose."""
    with mp_pose.Pose(static_image_mode=False, min_detection_confidence=0.5, min_tracking_confidence=0.5) as pose:
        results = pose.process(cv2.cvtColor(image, cv2.COLOR_BGR2RGB))
        if results.pose_landmarks:
            landmarks = results.pose_landmarks.landmark
            return [(lm.x, lm.y) for lm in landmarks]
        return []

def detect_weapons(image):
    """Dummy weapon detection for testing. Replace with a real weapon detector."""
    # Replace with your weapon detection model (e.g., YOLO)
    # Example: [(bbox, confidence)]
    return [((200, 300, 250, 350), 0.9)]  # Dummy weapon bounding box

def annotate_debug_info(image, faces, people_pose_landmarks, weapons):
    """Annotates the image with all keypoints, weapon positions, and face detections for debugging."""
    for face_bbox, label in faces:
        x, y, w, h = face_bbox
        cv2.rectangle(image, (x, y), (x + w, y + h), (0, 255, 0), 2)
        cv2.putText(image, label, (x, y - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 255, 0), 2)

    for idx, pose_keypoints in enumerate(people_pose_landmarks):
        image_h, image_w = image.shape[:2]
        for kp_idx, (x, y) in enumerate(pose_keypoints):
            abs_x, abs_y = int(x * image_w), int(y * image_h)
            cv2.circle(image, (abs_x, abs_y), 5, (255, 0, 0), -1)
            cv2.putText(image, f"{idx}_{kp_idx}", (abs_x + 5, abs_y), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 255, 255), 1)

    for weapon_bbox, _ in weapons:
        x1, y1, x2, y2 = weapon_bbox
        weapon_center = ((x1 + x2) // 2, (y1 + y2) // 2)
        cv2.rectangle(image, (x1, y1), (x2, y2), (0, 0, 255), 2)
        cv2.circle(image, weapon_center, 5, (0, 0, 255), -1)
        cv2.putText(image, "Weapon", (x1, y1 - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 0, 255), 2)

    return image

def find_weapon_holders(image, faces, people_pose_landmarks, weapons):
    """Identify the person closest to the weapon based on wrist distances."""
    image_h, image_w = image.shape[:2]
    weapon_holders = []

    for weapon_bbox, _ in weapons:
        x1, y1, x2, y2 = weapon_bbox
        weapon_center = ((x1 + x2) // 2, (y1 + y2) // 2)

        min_distance = float('inf')
        closest_person = None
        closest_hand = None

        for face, pose_keypoints in zip(faces, people_pose_landmarks):
            if not pose_keypoints:
                continue
            right_wrist = pose_keypoints[mp_pose.PoseLandmark.RIGHT_WRIST.value]
            left_wrist = pose_keypoints[mp_pose.PoseLandmark.LEFT_WRIST.value]

            right_wrist_abs = (int(right_wrist[0] * image_w), int(right_wrist[1] * image_h))
            left_wrist_abs = (int(left_wrist[0] * image_w), int(left_wrist[1] * image_h))

            right_distance = euclidean(weapon_center, right_wrist_abs)
            left_distance = euclidean(weapon_center, left_wrist_abs)

            if right_distance < min_distance:
                min_distance = right_distance
                closest_person = face[1]
                closest_hand = "Right Hand"
            if left_distance < min_distance:
                min_distance = left_distance
                closest_person = face[1]
                closest_hand = "Left Hand"

        if closest_person:
            weapon_holders.append((closest_person, closest_hand))

    return weapon_holders

# Main script
input_image_path = "single_with_weapon.jpeg"  # Replace with your image path
image = cv2.imread(input_image_path)

faces = detect_faces(image)
people_pose_landmarks = [detect_pose(image) for _ in faces]  # Match faces with pose landmarks
weapons = detect_weapons(image)

annotated_image_debug = annotate_debug_info(image.copy(), faces, people_pose_landmarks, weapons)
weapon_holders = find_weapon_holders(image, faces, people_pose_landmarks, weapons)

output_path = "debug_output_image.jpg"
cv2.imwrite(output_path, annotated_image_debug)

print(f"Weapon Holders: {weapon_holders}")
print(f"Debug output saved at {output_path}.")


Weapon Holders: [('Person_1', 'Right Hand')]
Debug output saved at debug_output_image.jpg.


In [30]:
weapon_holders

[]

In [63]:
import cv2
import numpy as np
from mediapipe import solutions as mp_solutions
from scipy.spatial.distance import euclidean
from tensorflow.keras.models import load_model

mp_pose = mp_solutions.pose
mp_face_detection = mp_solutions.face_detection

# Load the face recognition model and labels
face_model_path = "facerecognition_finalmodel.h5"
face_model = load_model(face_model_path)
face_labels = np.load('class_indices.npy', allow_pickle=True).item()

def detect_faces(image):
    """Detect faces in the image using MediaPipe Face Detection."""
    with mp_face_detection.FaceDetection(model_selection=0, min_detection_confidence=0.5) as face_detection:
        results = face_detection.process(cv2.cvtColor(image, cv2.COLOR_BGR2RGB))
        faces = []
        if results.detections:
            for detection in results.detections:
                bbox = detection.location_data.relative_bounding_box
                x, y, w, h = (int(bbox.xmin * image.shape[1]),
                              int(bbox.ymin * image.shape[0]),
                              int(bbox.width * image.shape[1]),
                              int(bbox.height * image.shape[0]))
                face_crop = image[y:y+h, x:x+w]
                face_label = recognize_face(face_crop)
                faces.append(((x, y, w, h), face_label))
        return faces

def recognize_face(face_crop):
    """Predict the identity of a detected face using the face recognition model."""
    try:
        face_resized = cv2.resize(face_crop, (100, 100))  # Resize for the model
        face_array = np.expand_dims(face_resized / 255.0, axis=0)  # Normalize and add batch dimension
        prediction = face_model.predict(face_array)
        label_index = np.argmax(prediction)
        confidence = np.max(prediction)
        print(label_index)
        if confidence > 0.5:  # Confidence threshold
            return list(face_labels.keys())[label_index]
        else:
            return "Unknown"
    except Exception as e:
        return "Error"

def detect_pose(image):
    """Detect pose landmarks in the image using MediaPipe Pose."""
    with mp_pose.Pose(static_image_mode=False, min_detection_confidence=0.5, min_tracking_confidence=0.5) as pose:
        results = pose.process(cv2.cvtColor(image, cv2.COLOR_BGR2RGB))
        if results.pose_landmarks:
            landmarks = results.pose_landmarks.landmark
            return [(lm.x, lm.y) for lm in landmarks]
        return []

def detect_weapons(image):
    """Dummy weapon detection for testing. Replace with a real weapon detector."""
    return [((200, 300, 250, 350), 0.9)]  # Dummy weapon bounding box

def annotate_debug_info(image, faces, people_pose_landmarks, weapons):
    """Annotates the image with all keypoints, weapon positions, and face detections for debugging."""
    for face_bbox, label in faces:
        x, y, w, h = face_bbox
        cv2.rectangle(image, (x, y), (x + w, y + h), (0, 255, 0), 2)
        cv2.putText(image, label, (x, y - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 255, 0), 2)

    for idx, pose_keypoints in enumerate(people_pose_landmarks):
        image_h, image_w = image.shape[:2]
        for kp_idx, (x, y) in enumerate(pose_keypoints):
            abs_x, abs_y = int(x * image_w), int(y * image_h)
            cv2.circle(image, (abs_x, abs_y), 5, (255, 0, 0), -1)
            cv2.putText(image, f"{idx}_{kp_idx}", (abs_x + 5, abs_y), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 255, 255), 1)

    for weapon_bbox, _ in weapons:
        x1, y1, x2, y2 = weapon_bbox
        weapon_center = ((x1 + x2) // 2, (y1 + y2) // 2)
        cv2.rectangle(image, (x1, y1), (x2, y2), (0, 0, 255), 2)
        cv2.circle(image, weapon_center, 5, (0, 0, 255), -1)
        cv2.putText(image, "Weapon", (x1, y1 - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 0, 255), 2)

    return image

def find_weapon_holders(image, faces, people_pose_landmarks, weapons):
    """Identify the person closest to the weapon based on wrist distances."""
    image_h, image_w = image.shape[:2]
    weapon_holders = []

    for weapon_bbox, _ in weapons:
        x1, y1, x2, y2 = weapon_bbox
        weapon_center = ((x1 + x2) // 2, (y1 + y2) // 2)

        min_distance = float('inf')
        closest_person = None
        closest_hand = None

        for face, pose_keypoints in zip(faces, people_pose_landmarks):
            if not pose_keypoints:
                continue
            right_wrist = pose_keypoints[mp_pose.PoseLandmark.RIGHT_WRIST.value]
            left_wrist = pose_keypoints[mp_pose.PoseLandmark.LEFT_WRIST.value]

            right_wrist_abs = (int(right_wrist[0] * image_w), int(right_wrist[1] * image_h))
            left_wrist_abs = (int(left_wrist[0] * image_w), int(left_wrist[1] * image_h))

            right_distance = euclidean(weapon_center, right_wrist_abs)
            left_distance = euclidean(weapon_center, left_wrist_abs)

            if right_distance < min_distance:
                min_distance = right_distance
                closest_person = face[1]
                closest_hand = "Right Hand"
            if left_distance < min_distance:
                min_distance = left_distance
                closest_person = face[1]
                closest_hand = "Left Hand"

        if closest_person:
            weapon_holders.append((closest_person, closest_hand))

    return weapon_holders

# Main script
input_image_path = "465679287-1617663771_jpeg_jpg.rf.81225b8bf367b2295c4adb571072a999.jpg"  # Replace with your image path
image = cv2.imread(input_image_path)

faces = detect_faces(image)
people_pose_landmarks = [detect_pose(image) for _ in faces]  # Match faces with pose landmarks
weapons = detect_weapons(image)

annotated_image_debug = annotate_debug_info(image.copy(), faces, people_pose_landmarks, weapons)
weapon_holders = find_weapon_holders(image, faces, people_pose_landmarks, weapons)

output_path = "debug_output_image123.jpg"
cv2.imwrite(output_path, annotated_image_debug)

print(f"Weapon Holders: {weapon_holders}")
print(f"Debug output saved at {output_path}.")


1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 937ms/step
3
Weapon Holders: [('sreekanth', 'Left Hand')]
Debug output saved at debug_output_image123.jpg.


In [49]:
import cv2
import numpy as np
from mediapipe import solutions as mp_solutions
from scipy.spatial.distance import euclidean
from tensorflow.keras.models import load_model

mp_pose = mp_solutions.pose
mp_face_detection = mp_solutions.face_detection

# Load the face recognition model and labels
face_model_path = "facerecognition_finalmodel.h5"
face_model = load_model(face_model_path)
face_labels = np.load('class_indices.npy', allow_pickle=True).item()

def detect_faces(image):
    """Detect faces in the image using MediaPipe Face Detection."""
    with mp_face_detection.FaceDetection(model_selection=0, min_detection_confidence=0.5) as face_detection:
        results = face_detection.process(cv2.cvtColor(image, cv2.COLOR_BGR2RGB))
        faces = []
        if results.detections:
            for detection in results.detections:
                bbox = detection.location_data.relative_bounding_box
                x, y, w, h = (int(bbox.xmin * image.shape[1]),
                              int(bbox.ymin * image.shape[0]),
                              int(bbox.width * image.shape[1]),
                              int(bbox.height * image.shape[0]))
                face_crop = image[y:y+h, x:x+w]
                face_label = recognize_face(face_crop)
                faces.append(((x, y, w, h), face_label))
        return faces

def recognize_face(face_crop):
    """Predict the identity of a detected face using the face recognition model."""
    try:
        face_resized = cv2.resize(face_crop, (100, 100))  # Resize for the model
        face_array = np.expand_dims(face_resized / 255.0, axis=0)  # Normalize and add batch dimension
        prediction = face_model.predict(face_array)
        label_index = np.argmax(prediction)
        confidence = np.max(prediction)
        print(label_index)
        if confidence > 0.5:  # Confidence threshold
            return list(face_labels.keys())[label_index]
        else:
            return "Unknown"
    except Exception as e:
        return "Error"

def detect_pose(image):
    """Detect pose landmarks in the image using MediaPipe Pose."""
    with mp_pose.Pose(static_image_mode=False, min_detection_confidence=0.5, min_tracking_confidence=0.5) as pose:
        results = pose.process(cv2.cvtColor(image, cv2.COLOR_BGR2RGB))
        if results.pose_landmarks:
            landmarks = results.pose_landmarks.landmark
            return [(lm.x, lm.y) for lm in landmarks]
        return []

def detect_weapons(image):
    """Dummy weapon detection for testing. Replace with a real weapon detector."""
    return [((200, 300, 250, 350), 0.9)]  # Dummy weapon bounding box

def annotate_debug_info(image, faces, people_pose_landmarks, weapons):
    """Annotates the image with all keypoints, weapon positions, and face detections for debugging."""
    for face_bbox, label in faces:
        x, y, w, h = face_bbox
        cv2.rectangle(image, (x, y), (x + w, y + h), (0, 255, 0), 2)
        cv2.putText(image, label, (x, y - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 255, 0), 2)

    for idx, pose_keypoints in enumerate(people_pose_landmarks):
        image_h, image_w = image.shape[:2]
        for kp_idx, (x, y) in enumerate(pose_keypoints):
            abs_x, abs_y = int(x * image_w), int(y * image_h)
            cv2.circle(image, (abs_x, abs_y), 5, (255, 0, 0), -1)
            cv2.putText(image, f"{idx}_{kp_idx}", (abs_x + 5, abs_y), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 255, 255), 1)

    for weapon_bbox, _ in weapons:
        x1, y1, x2, y2 = weapon_bbox
        weapon_center = ((x1 + x2) // 2, (y1 + y2) // 2)
        cv2.rectangle(image, (x1, y1), (x2, y2), (0, 0, 255), 2)
        cv2.circle(image, weapon_center, 5, (0, 0, 255), -1)
        cv2.putText(image, "Weapon", (x1, y1 - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 0, 255), 2)

    return image

def find_weapon_holders(image, faces, people_pose_landmarks, weapons):
    """Identify the person closest to the weapon based on wrist distances."""
    image_h, image_w = image.shape[:2]
    weapon_holders = []

    for weapon_bbox, _ in weapons:
        x1, y1, x2, y2 = weapon_bbox
        weapon_center = ((x1 + x2) // 2, (y1 + y2) // 2)

        min_distance = float('inf')
        closest_person = None
        closest_hand = None

        for face, pose_keypoints in zip(faces, people_pose_landmarks):
            if not pose_keypoints:
                continue
            right_wrist = pose_keypoints[mp_pose.PoseLandmark.RIGHT_WRIST.value]
            left_wrist = pose_keypoints[mp_pose.PoseLandmark.LEFT_WRIST.value]

            right_wrist_abs = (int(right_wrist[0] * image_w), int(right_wrist[1] * image_h))
            left_wrist_abs = (int(left_wrist[0] * image_w), int(left_wrist[1] * image_h))

            right_distance = euclidean(weapon_center, right_wrist_abs)
            left_distance = euclidean(weapon_center, left_wrist_abs)

            if right_distance < min_distance:
                min_distance = right_distance
                closest_person = face[1]
                closest_hand = "Right Hand"
            if left_distance < min_distance:
                min_distance = left_distance
                closest_person = face[1]
                closest_hand = "Left Hand"

        if closest_person:
            weapon_holders.append((closest_person, closest_hand))

    return weapon_holders

# Main script
input_image_path = "single_with_weapon.jpeg"  # Replace with your image path
image = cv2.imread(input_image_path)

faces = detect_faces(image)
people_pose_landmarks = [detect_pose(image) for _ in faces]  # Match faces with pose landmarks
weapons = detect_weapons(image)

annotated_image_debug = annotate_debug_info(image.copy(), faces, people_pose_landmarks, weapons)
weapon_holders = find_weapon_holders(image, faces, people_pose_landmarks, weapons)

output_path = "debug_output_image.jpg"
cv2.imwrite(output_path, annotated_image_debug)

print(f"Weapon Holders: {weapon_holders}")
print(f"Debug output saved at {output_path}.")


1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 947ms/step
2
Weapon Holders: [('others', 'Right Hand')]
Debug output saved at debug_output_image.jpg.


In [60]:
import cv2
import numpy as np
from mediapipe import solutions as mp_solutions
from scipy.spatial.distance import euclidean
from tensorflow.keras.models import load_model

mp_pose = mp_solutions.pose
mp_face_detection = mp_solutions.face_detection

# Load the face recognition model and labels
face_model_path = "facerecognition_finalmodel.h5"
face_model = load_model(face_model_path)
face_labels = np.load('class_indices.npy', allow_pickle=True).item()

def detect_faces(image):
    """Detect faces in the image using MediaPipe Face Detection."""
    with mp_face_detection.FaceDetection(model_selection=0, min_detection_confidence=0.5) as face_detection:
        results = face_detection.process(cv2.cvtColor(image, cv2.COLOR_BGR2RGB))
        faces = []
        if results.detections:
            for detection in results.detections:
                bbox = detection.location_data.relative_bounding_box
                x, y, w, h = (int(bbox.xmin * image.shape[1]),
                              int(bbox.ymin * image.shape[0]),
                              int(bbox.width * image.shape[1]),
                              int(bbox.height * image.shape[0]))
                face_crop = image[y:y+h, x:x+w]
                face_label = recognize_face(face_crop)
                faces.append(((x, y, w, h), face_label))
        return faces

def recognize_face(face_crop):
    """Predict the identity of a detected face using the face recognition model."""
    try:
        face_resized = cv2.resize(face_crop, (100, 100))  # Resize for the model
        face_array = np.expand_dims(face_resized / 255.0, axis=0)  # Normalize and add batch dimension
        prediction = face_model.predict(face_array)
        label_index = np.argmax(prediction)
        confidence = np.max(prediction)
        if confidence > 0.5:  # Confidence threshold
            return list(face_labels.keys())[label_index]
        else:
            return "Unknown"
    except Exception as e:
        return "Error"

def detect_pose(image):
    """Detect pose landmarks in the image using MediaPipe Pose."""
    with mp_pose.Pose(static_image_mode=False, min_detection_confidence=0.5, min_tracking_confidence=0.5) as pose:
        results = pose.process(cv2.cvtColor(image, cv2.COLOR_BGR2RGB))
        if results.pose_landmarks:
            landmarks = results.pose_landmarks.landmark
            return [(lm.x, lm.y) for lm in landmarks]
        return []

def detect_weapons(image):
    """Dummy weapon detection for testing. Replace with a real weapon detector."""
    return [((200, 300, 250, 350), 0.9)]  # Dummy weapon bounding box

def annotate_debug_info(image, faces, people_pose_landmarks, weapons):
    """Annotates the image with all keypoints, weapon positions, and face detections for debugging."""
    for face_bbox, label in faces:
        x, y, w, h = face_bbox
        cv2.rectangle(image, (x, y), (x + w, y + h), (0, 255, 0), 2)
        cv2.putText(image, label, (x, y - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 255, 0), 2)

    for weapon_bbox, _ in weapons:
        x1, y1, x2, y2 = weapon_bbox
        weapon_center = ((x1 + x2) // 2, (y1 + y2) // 2)
        cv2.rectangle(image, (x1, y1), (x2, y2), (0, 0, 255), 2)
        cv2.circle(image, weapon_center, 5, (0, 0, 255), -1)
        cv2.putText(image, "Weapon", (x1, y1 - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 0, 255), 2)

    return image

def find_weapon_holders(image, faces, people_pose_landmarks, weapons):
    """Identify the person closest to the weapon based on wrist distances."""
    image_h, image_w = image.shape[:2]
    weapon_holders = []

    for weapon_bbox, _ in weapons:
        x1, y1, x2, y2 = weapon_bbox
        weapon_center = ((x1 + x2) // 2, (y1 + y2) // 2)

        min_distance = float('inf')
        closest_person = None
        closest_hand = None

        for face, pose_keypoints in zip(faces, people_pose_landmarks):
            if not pose_keypoints:
                continue
            right_wrist = pose_keypoints[mp_pose.PoseLandmark.RIGHT_WRIST.value]
            left_wrist = pose_keypoints[mp_pose.PoseLandmark.LEFT_WRIST.value]

            right_wrist_abs = (int(right_wrist[0] * image_w), int(right_wrist[1] * image_h))
            left_wrist_abs = (int(left_wrist[0] * image_w), int(left_wrist[1] * image_h))

            right_distance = euclidean(weapon_center, right_wrist_abs)
            left_distance = euclidean(weapon_center, left_wrist_abs)

            if right_distance < min_distance:
                min_distance = right_distance
                closest_person = face[1]
                closest_hand = "Right Hand"
            if left_distance < min_distance:
                min_distance = left_distance
                closest_person = face[1]
                closest_hand = "Left Hand"

        if closest_person:
            weapon_holders.append((closest_person, closest_hand))

    return weapon_holders

# Main script
input_image_path = "single_with_weapon.jpeg"  # Replace with your image path
image = cv2.imread(input_image_path)

faces = detect_faces(image)
people_pose_landmarks = [detect_pose(image) for _ in faces]  # Match faces with pose landmarks
weapons = detect_weapons(image)

annotated_image_debug = annotate_debug_info(image.copy(), faces, people_pose_landmarks, weapons)
weapon_holders = find_weapon_holders(image, faces, people_pose_landmarks, weapons)

output_path = "debug_output_image.jpg"
cv2.imwrite(output_path, annotated_image_debug)

print(f"Weapon Holders: {weapon_holders}")
print(f"Debug output saved at {output_path}.")


1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 942ms/step
Weapon Holders: [('others', 'Right Hand')]
Debug output saved at debug_output_image.jpg.


In [62]:
import cv2
import numpy as np
from mediapipe import solutions as mp_solutions
from scipy.spatial.distance import euclidean
from tensorflow.keras.models import load_model

mp_pose = mp_solutions.pose
mp_face_detection = mp_solutions.face_detection

# Load the face recognition model and labels
face_model_path = "facerecognition_finalmodel.h5"
face_model = load_model(face_model_path)
face_labels = np.load('class_indices.npy', allow_pickle=True).item()

def detect_faces(image):
    """Detect faces in the image using MediaPipe Face Detection."""
    with mp_face_detection.FaceDetection(model_selection=0, min_detection_confidence=0.5) as face_detection:
        results = face_detection.process(cv2.cvtColor(image, cv2.COLOR_BGR2RGB))
        faces = []
        if results.detections:
            for detection in results.detections:
                bbox = detection.location_data.relative_bounding_box
                x, y, w, h = (int(bbox.xmin * image.shape[1]),
                              int(bbox.ymin * image.shape[0]),
                              int(bbox.width * image.shape[1]),
                              int(bbox.height * image.shape[0]))
                face_crop = image[y:y+h, x:x+w]
                face_label = recognize_face(face_crop)
                faces.append(((x, y, w, h), face_label))
        return faces

def recognize_face(face_crop):
    """Predict the identity of a detected face using the face recognition model."""
    try:
        face_resized = cv2.resize(face_crop, (100, 100))  # Resize for the model
        face_array = np.expand_dims(face_resized / 255.0, axis=0)  # Normalize and add batch dimension
        prediction = face_model.predict(face_array)
        label_index = np.argmax(prediction)
        confidence = np.max(prediction)
        if confidence > 0.5:  # Confidence threshold
            return list(face_labels.keys())[label_index]
        else:
            return "Unknown"
    except Exception as e:
        return "Error"

def detect_pose(image):
    """Detect pose landmarks in the image using MediaPipe Pose."""
    with mp_pose.Pose(static_image_mode=False, min_detection_confidence=0.5, min_tracking_confidence=0.5) as pose:
        results = pose.process(cv2.cvtColor(image, cv2.COLOR_BGR2RGB))
        if results.pose_landmarks:
            landmarks = results.pose_landmarks.landmark
            return [(lm.x, lm.y) for lm in landmarks]
        return []

def detect_weapons(image):
    """Dummy weapon detection for testing. Replace with a real weapon detector."""
    return [((200, 300, 250, 350), 0.9)]  # Dummy weapon bounding box

def annotate_debug_info(image, faces, people_pose_landmarks, weapons):
    """Annotates the image with all keypoints, weapon positions, and face detections for debugging."""
    for face_bbox, label in faces:
        x, y, w, h = face_bbox
        cv2.rectangle(image, (x, y), (x + w, y + h), (0, 255, 0), 2)
        cv2.putText(image, label, (x, y - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 255, 0), 2)

    for weapon_bbox, _ in weapons:
        x1, y1, x2, y2 = weapon_bbox
        weapon_center = ((x1 + x2) // 2, (y1 + y2) // 2)
        cv2.rectangle(image, (x1, y1), (x2, y2), (0, 0, 255), 2)
        cv2.circle(image, weapon_center, 5, (0, 0, 255), -1)
        cv2.putText(image, "Weapon", (x1, y1 - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 0, 255), 2)

    return image

def find_weapon_holders(image, faces, people_pose_landmarks, weapons):
    """Identify the person closest to the weapon based on wrist distances."""
    image_h, image_w = image.shape[:2]
    weapon_holders = []

    for weapon_bbox, _ in weapons:
        x1, y1, x2, y2 = weapon_bbox
        weapon_center = ((x1 + x2) // 2, (y1 + y2) // 2)

        min_distance = float('inf')
        closest_person = None
        closest_hand = None

        for face, pose_keypoints in zip(faces, people_pose_landmarks):
            if not pose_keypoints:
                continue
            right_wrist = pose_keypoints[mp_pose.PoseLandmark.RIGHT_WRIST.value]
            left_wrist = pose_keypoints[mp_pose.PoseLandmark.LEFT_WRIST.value]

            right_wrist_abs = (int(right_wrist[0] * image_w), int(right_wrist[1] * image_h))
            left_wrist_abs = (int(left_wrist[0] * image_w), int(left_wrist[1] * image_h))

            right_distance = euclidean(weapon_center, right_wrist_abs)
            left_distance = euclidean(weapon_center, left_wrist_abs)

            if right_distance < min_distance:
                min_distance = right_distance
                closest_person = face[1]
                closest_hand = "Right Hand"
            if left_distance < min_distance:
                min_distance = left_distance
                closest_person = face[1]
                closest_hand = "Left Hand"

        if closest_person:
            weapon_holders.append((closest_person, closest_hand))

    return weapon_holders

# Main script
input_image_path = "465679287-1617663771_jpeg_jpg.rf.81225b8bf367b2295c4adb571072a999.jpg"  # Replace with your image path
image = cv2.imread(input_image_path)

faces = detect_faces(image)
people_pose_landmarks = [detect_pose(image) for _ in faces]  # Match faces with pose landmarks
weapons = detect_weapons(image)

annotated_image_debug = annotate_debug_info(image.copy(), faces, people_pose_landmarks, weapons)
weapon_holders = find_weapon_holders(image, faces, people_pose_landmarks, weapons)

output_path = "debug_output_image.jpg"
cv2.imwrite(output_path, annotated_image_debug)

print(f"Weapon Holders: {weapon_holders}")
print(f"Debug output saved at {output_path}.")


1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 939ms/step
Weapon Holders: [('sreekanth', 'Left Hand')]
Debug output saved at debug_output_image.jpg.
